In [ ]:
!pip install tensorflow scikit-learn pandas numpy -q

In [ ]:
# !pip uninstall -y wandb
# !pip install wandb

In [ ]:
import wandb.integration.keras as wk
print(dir(wk))

['WandbEvalCallback', 'WandbMetricsLogger', 'WandbModelCheckpoint', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'callbacks', 'keras']


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import wandb
from wandb.integration.keras import WandbMetricsLogger
from wandb.integration.keras import WandbModelCheckpoint
import joblib

In [ ]:
df = pd.read_csv('/content/tehranhouses.csv')
df['Area'] = df['Area'].apply(lambda x: str(x).replace(',', ''))
df["Area"] = pd.to_numeric(df["Area"], errors='coerce')
df = df.dropna()

In [ ]:
X = df.drop(columns=['Price', 'Price(USD)'])
y = df['Price']
X = pd.get_dummies(X, columns=['Address'], drop_first=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
wandb.init(project="tehran-house-colab", config={
    "learning_rate": 0.001,
    "epochs": 100,
    "batch_size": 32,
    "architecture": "MLP-Deep",
    "hidden_layers": [128, 64, 32]
})
config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adel-kefayat8120 (adel-kefayat8120-maktabkhooneh) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
model = Sequential([
    Dense(config.hidden_layers[0], activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(config.hidden_layers[1], activation='relu'),
    Dropout(0.1),
    Dense(config.hidden_layers[2], activation='relu'),
    Dense(1)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.learning_rate),
    loss='mse',
    metrics=['mae']
)

In [ ]:
wandb_callbacks = [
    WandbMetricsLogger(log_freq="epoch"), # ثبت متغیرها به ازای هر Epoch
    WandbModelCheckpoint(filepath="best_model.keras", save_best_only=True) # ذخیره و آپلود خودکار بهترین مدل
]

wandb: WARNING When using `save_best_only`, ensure that the `filepath` argument contains formatting placeholders like `{epoch:02d}` or `{batch:02d}`. This ensures correct interpretation of the logged artifacts.


In [ ]:
model.fit(
    X_train_scaled, y_train,
    epochs=config.epochs,
    batch_size=config.batch_size,
    validation_data=(X_test_scaled, y_test),
    callbacks=wandb_callbacks
)

Epoch 1/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 16s 76ms/step - loss: 90740900237376749568.0000 - mae: 5324685824.0000 - val_loss: 111449198991933177856.0000 - val_mae: 5600719360.0000
Epoch 2/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 90740847460818616320.0000 - mae: 5324685312.0000 - val_loss: 111449163807561089024.0000 - val_mae: 5600715776.0000
Epoch 3/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 90740697927237238784.0000 - mae: 5324675584.0000 - val_loss: 111448767983375089664.0000 - val_mae: 5600694784.0000
Epoch 4/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 90739976647609417728.0000 - mae: 5324630528.0000 - val_loss: 111447510142072913920.0000 - val_mae: 5600621568.0000
Epoch 5/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 90738032711051509760.0000 - mae: 5324509696.0000 - val_loss: 111444413917329096704.0000 - val_mae: 5600446464.0000
Epoch 6/100
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 90733740217656672256.0000 - mae: 5324254720.0000 - val_loss: 11143818

In [ ]:
model.save("deep_house_model.h5")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X_train.columns.tolist(), "features_columns.pkl")

['features_columns.pkl']

In [ ]:
print("--- آموزش با موفقیت تمام شد و Artifactها در WandB ثبت شدند ---")
wandb.finish()

--- آموزش با موفقیت تمام شد و Artifactها در WandB ثبت شدند ---


epoch/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,███████████▇▇▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁
epoch/mae,███████████▇▇▇▇▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch/val_loss,████████████▇▇▇▇▇▆▆▆▆▆▅▅▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁
epoch/val_mae,██████████▇▇▇▇▆▆▅▅▄▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/epoch,99
epoch/learning_rate,0.001
epoch/loss,4.043452691732287e+19
epoch/mae,2820186624.0
epoch/val_loss,5.430503762553104e+19


In [ ]:
%%writefile app.py
import gradio as gr
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import os

# ۱. تابع بارگذاری فایل‌های مدل و پیش‌پردازش
def load_assets():
    model_path = "deep_house_model.h5"
    scaler_path = "scaler.pkl"
    columns_path = "features_columns.pkl"

    if not (os.path.exists(model_path) and os.path.exists(scaler_path) and os.path.exists(columns_path)):
        return None, None, None

    model = tf.keras.models.load_model(model_path)
    scaler = joblib.load(scaler_path)
    columns = joblib.load(columns_path)
    return model, scaler, columns

model, scaler, columns = load_assets()

# ۲. تابع اصلی پیش‌بینی قیمت مسکن برای رابط کاربری Gradio
def predict_house_price(area, rooms, parking, warehouse, elevator, address):
    if model is None:
        return "⚠️ خطای سیستم: فایل‌های مدل یافت نشدند!"

    # ساخت یک دیتافریم با یک ردیف مشابه ساختار زمان آموزش
    input_df = pd.DataFrame(0, index=[0], columns=columns)
    input_df['Area'] = float(area)
    input_df['Room'] = int(rooms)
    input_df['Parking'] = 1 if parking else 0
    input_df['Warehouse'] = 1 if warehouse else 0
    input_df['Elevator'] = 1 if elevator else 0

    # اعمال متغیر One-Hot محله انتخاب شده
    address_col = f"Address_{address}"
    if address_col in input_df.columns:
        input_df[address_col] = 1

    # نرمال‌سازی داده ورودی و پیش‌بینی توسط شبکه عصبی
    input_scaled = scaler.transform(input_df)
    prediction = model.predict(input_scaled)[0][0]

    if prediction < 0:
        return "❌ ورودی‌های نامعتبر! لطفاً متراژ یا محله را تغییر دهید."

    return f"💰 قیمت تخمینی هوش مصنوعی: {prediction:,.0f} تومان"

# ۳. طراحی ظاهر برنامه (UI) با کامپوننت‌های Gradio
if columns is not None:
    addresses = sorted([col.replace("Address_", "") for col in columns if "Address_" in col])
else:
    addresses = []

with gr.Blocks(theme=gr.themes.Soft(primary_hue="purple", secondary_hue="indigo")) as demo:
    gr.Markdown("# 🚀 سامانه هوشمند تخمین قیمت مسکن تهران (Gradio)")
    gr.Markdown("مشخصات ملک را در کادرهای زیر وارد کنید تا شبکه عصبی عمیق ارزش آن را محاسبه کند.")

    with gr.Row():
        with gr.Column():
            area = gr.Number(label="📐 متراژ بنا (متر مربع)", value=85)
            rooms = gr.Slider(minimum=0, maximum=5, step=1, label="🛏️ تعداد اتاق خواب", value=2)
            address = gr.Dropdown(choices=addresses, label="📍 انتخاب محله", value=addresses[0] if addresses else None)

            gr.Markdown("**🛠️ امکانات ملک:**")
            parking = gr.Checkbox(label="🅿️ پارکینگ دارد", value=True)
            warehouse = gr.Checkbox(label="📦 انباری دارد", value=True)
            elevator = gr.Checkbox(label="🔼 آسانسور دارد", value=True)

            submit_btn = gr.Button("🔮 محاسبه قیمت", variant="primary")

        with gr.Column():
            output_text = gr.Textbox(label="📡 خروجی هسته پردازش مدل", placeholder="منتظر ورود اطلاعات...", interactive=False)
            gr.HTML('<img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExM3N6Z3Z5cmZqamg3OHg0OHR3Z3d5cW0xa210N3N5ZzBndmxiN3Z6ZCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/X78rWLUfLs6A79MzQu/giphy.gif" style="width:100%; border-radius:10px;">')

    submit_btn.click(
        fn=predict_house_price,
        inputs=[area, rooms, parking, warehouse, elevator, address],
        outputs=output_text
    )

demo.launch()

Overwriting app.py
